<a href="https://colab.research.google.com/github/ancestor9/2026_Fall_Learning-Langchain-AI-Agent/blob/main/CH6_Agent_Architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from google.colab import userdata

# Groq API 키 설정
YOUR_API_KEY = userdata.get('groq')
os.environ["GROQ_API_KEY"] = YOUR_API_KEY

In [2]:
!pip install -qU langchain-groq
!pip install -qU langchain-community
!pip install duckduckgo-search -q
!pip install -U ddgs

In [5]:
# @title **Building a LangGraph Agent**

import ast
from typing import Annotated, TypedDict
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langgraph.graph import START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition


@tool
def calculator(query: str) -> str:
    """A simple calculator tool. Input should be a mathematical expression."""
    try:
        return str(eval(query))
    except Exception as e:
        return f"Error evaluating expression: {e}"


search = DuckDuckGoSearchRun()
tools = [search, calculator]
model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0).bind_tools(tools)

class State(TypedDict):
    messages: Annotated[list, add_messages]


def model_node(state: State) -> State:
    res = model.invoke(state["messages"])
    return {"messages": res}

builder = StateGraph(State)

# LangGraph의 tools_condition은 "루프 탈출용 output 도구" 역할을 맡아 처리
# LLM이 도구 호출이 필요하다고 판단하면: AIMessage에 tool_calls 정보를 담아 반환하고, tools_condition은 이를 감지해 tools 노드로 이동
# LLM이 충분한 정보를 얻어 최종 답을 작성하면: tool_calls 없이 일반 텍스트 답변만 반환하면 tools_condition은 "아, 이제 도구 안 쓰는구나!" 하고
# 자동으로 END(종료)로 연결, 즉, 별도의 output 도구를 정의하지 않아도 tools_condition이 루프 탈출 조건(Stop Condition) 역할을 수행

# 1. 노드(단계) 등록
builder.add_node("model", model_node)      # LLM이 판단하는 노드
builder.add_node("tools", ToolNode(tools))  # 외부 도구를 실제 실행하는 노드

# 2. 흐름(Edge) 연결
builder.add_edge(START, "model")           # 시작하면 먼저 LLM에게 전달
builder.add_conditional_edges("model", tools_condition) # LLM 결과에 따른 조건부 분기
builder.add_edge("tools", "model")         # 도구 실행 후 다시 LLM으로 돌아감 (Loop!)

graph = builder.compile()

# Example usage

input = {
    "messages": [
        HumanMessage(
            "How old was the 30th president of the United States when he died?"
        )
    ]
}

for c in graph.stream(input):
    print(c)

{'model': {'messages': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'dmbd18rmq', 'function': {'arguments': '{"query":"age of 30th president of the United States at death"}', 'name': 'duckduckgo_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 341, 'total_tokens': 369, 'completion_time': 0.057438475, 'completion_tokens_details': None, 'prompt_time': 0.017481165, 'prompt_tokens_details': None, 'queue_time': 0.051552432, 'total_time': 0.07491964}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f8760-5377-7df0-8cdb-d590fb84a9f4-0', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'age of 30th president of the United States at death'}, 'id': 'dmbd18rmq', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 341, 'output_tokens':

In [8]:
# @title **Always Calling a Tool First**

from uuid import uuid4

from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.messages import AIMessage, HumanMessage, ToolCall
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langgraph.graph import START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition


@tool
def calculator(query: str) -> str:
    """A simple calculator tool. Input should be a mathematical expression."""
    try:
        return str(eval(query))
    except Exception as e:
        return f"Error evaluating expression: {e}"


search = DuckDuckGoSearchRun()
tools = [search, calculator]
model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0).bind_tools(tools)


class State(TypedDict):
    messages: Annotated[list, add_messages]


def model_node(state: State) -> State:
    res = model.invoke(state["messages"])
    return {"messages": res}


def first_model(state: State) -> State:
    query = state["messages"][-1].content
    search_tool_call = ToolCall(
        name="duckduckgo_search", args={"query": query}, id=uuid4().hex
    )
    return {"messages": AIMessage(content="", tool_calls=[search_tool_call])}

builder = StateGraph(State)

#
builder.add_node("first_model", first_model)

builder.add_node("model", model_node)
builder.add_node("tools", ToolNode(tools))

#
builder.add_edge(START, "first_model")
builder.add_edge("first_model", "tools")
#-----------------------------------------

builder.add_conditional_edges("model", tools_condition)
builder.add_edge("tools", "model")

graph = builder.compile()

# Example usage
input = {
    "messages": [
        HumanMessage(
            "How old was the 30th president of the United States when he died?"
        )
    ]
}

for c in graph.stream(input):
    print(c)

{'first_model': {'messages': AIMessage(content='', additional_kwargs={}, response_metadata={}, id='ae8ac37b-4c1c-4e20-825d-2ddafdb615b8', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'How old was the 30th president of the United States when he died?'}, 'id': '2d52a884b8774925bea59c692b39903c', 'type': 'tool_call'}], invalid_tool_calls=[])}}
{'tools': {'messages': [ToolMessage(content='3 days ago - Calvin Coolidge (born John Calvin Coolidge Jr.; /ˈkuːlɪdʒ/ KOOL-ij; July 4, 1872 – January 5, 1933) was the 30th president of the United States, serving from 1923 to 1929. A Republican lawyer from Massachusetts, he served from 1921 to 1923 as the 29th vice president, under President Warren G. March 21, 2026 - The following is a list of presidents ... statistics. Of the 45 people who have served as President of the United States since the office came into existence in 1789, 40 have died – eight of them while in office. The oldest president at the time of death was Jimmy Carter, 

In [9]:
input = {
 "messages": [
 HumanMessage("""How old was the 30th president of the United States
 when he died?""")
 ]
}
for c in graph.stream(input):
    print(c)


{'first_model': {'messages': AIMessage(content='', additional_kwargs={}, response_metadata={}, id='98e77989-d645-4bf3-8a09-ee4310e79792', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'How old was the 30th president of the United States\n when he died?'}, 'id': '49ede279f19a4bfb9aa66c289a843f52', 'type': 'tool_call'}], invalid_tool_calls=[])}}
{'tools': {'messages': [ToolMessage(content='Calvin Coolidge (born John Calvin Coolidge Jr.; [1] / ˈkuːlɪdʒ / KOOL-ij; July 4, 1872 – January 5, 1933) was the 30th president of the United States, serving from 1923 to 1929. John F. Kennedy, assassinated at the age of 46 years, 177 days, was the youngest to have died in office; the youngest to have died by natural causes was James K. Polk, who died of cholera at the age of 53 years, 225 days. Calvin Coolidge was the 30th president of the United States (1923–29). He acceded to the presidency after the death in office of Warren G. Harding, just as the Harding scandals were coming to lig

In [10]:
!pip install -qU langchain-huggingface

In [12]:
# @title **Dealing with Many Tools**

import ast
from typing import Annotated, TypedDict

from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_core.vectorstores.in_memory import InMemoryVectorStore
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings # Updated import

from langgraph.graph import START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition


@tool
def calculator(query: str) -> str:
    """A simple calculator tool. Input should be a mathematical expression."""
    return ast.literal_eval(query)


search = DuckDuckGoSearchRun()
tools = [search, calculator]

# HuggingFace 384차원 임베딩 모델 설정
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0).bind_tools(tools)

tools_retriever = InMemoryVectorStore.from_documents(
    [Document(tool.description, metadata={"name": tool.name}) for tool in tools],
    embeddings,
).as_retriever()


class State(TypedDict):
    messages: Annotated[list, add_messages]
    selected_tools: list[str]


def model_node(state: State) -> State:
    selected_tools = [tool for tool in tools if tool.name in state["selected_tools"]]
    res = model.bind_tools(selected_tools).invoke(state["messages"])
    return {"messages": res}


def select_tools(state: State) -> State:
    query = state["messages"][-1].content
    tool_docs = tools_retriever.invoke(query)
    return {"selected_tools": [doc.metadata["name"] for doc in tool_docs]}


builder = StateGraph(State)
builder.add_node("select_tools", select_tools)
builder.add_node("model", model_node)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "select_tools")
builder.add_edge("select_tools", "model")
builder.add_conditional_edges("model", tools_condition)
builder.add_edge("tools", "model")

graph = builder.compile()

# Example usage
input = {
    "messages": [
        HumanMessage(
            "How old was the 30th president of the United States when he died?"
        )
    ]
}

for c in graph.stream(input):
    print(c)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

{'select_tools': {'selected_tools': ['calculator', 'duckduckgo_search']}}
{'model': {'messages': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'm0zj0ff5r', 'function': {'arguments': '{"query":"age of 30th president of the United States at death"}', 'name': 'duckduckgo_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 341, 'total_tokens': 369, 'completion_time': 0.059856014, 'completion_tokens_details': None, 'prompt_time': 0.033591352, 'prompt_tokens_details': None, 'queue_time': 0.052313678, 'total_time': 0.093447366}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f87a8-4165-7583-b08a-c50f9c507b67-0', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'age of 30th president of the United States at death'}, 'id': 'm0zj0ff5r', 'type': 'tool_call'}], i

In [4]:
aaa

NameError: name 'aaa' is not defined

In [ ]:
# @title **Architecture #1: LLM Call**

from langchain_groq import ChatGroq
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage

model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

class State(TypedDict):
    # Messages have the type "list". The `add_messages`
    # function in the annotation defines how this state should
    # be updated (in this case, it appends new messages to the
    # list, rather than replacing the previous messages)
    messages: Annotated[list, add_messages]

def chatbot(state: State):
    answer = model.invoke(state["messages"])
    return {"messages": [answer]}

builder = StateGraph(State)

builder.add_node("chatbot", chatbot)

builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

graph = builder.compile()

# Example usage

input = {"messages": [HumanMessage("hi!")]}
for chunk in graph.stream(input):
    print(chunk)

In [ ]:
[chunk for chunk in graph.stream(input)]

In [ ]:
# @title **Architecture #2: Chain**

from typing import Annotated, TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

# useful to generate SQL query
model_low_temp = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.1)
# useful to generate natural language outputs
model_high_temp = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7)

class State(TypedDict):
    # to track conversation history
    messages: Annotated[list, add_messages]
    # input
    user_query: str
    # output
    sql_query: str
    sql_explanation: str

class Input(TypedDict):
    user_query: str

class Output(TypedDict):
    sql_query: str
    sql_explanation: str

generate_prompt = SystemMessage(
    "You are a helpful data analyst, who generates SQL queries for users based on their questions."
)

def generate_sql(state: State) -> State:
    user_message = HumanMessage(state["user_query"])
    messages = [generate_prompt, *state["messages"], user_message]
    res = model_low_temp.invoke(messages)
    return {
        "sql_query": res.content,
        # update conversation history
        "messages": [user_message, res],
    }

explain_prompt = SystemMessage(
    "You are a helpful data analyst, who explains SQL queries to users."
)

def explain_sql(state: State) -> State:
    messages = [
        explain_prompt,
        # contains user's query and SQL query from prev step
        *state["messages"],
    ]
    res = model_high_temp.invoke(messages)
    return {
        "sql_explanation": res.content,
        # update conversation history
        "messages": res,
    }

builder = StateGraph(State, input=Input, output=Output)
builder.add_node("generate_sql", generate_sql)
builder.add_node("explain_sql", explain_sql)
builder.add_edge(START, "generate_sql")
builder.add_edge("generate_sql", "explain_sql")
builder.add_edge("explain_sql", END)

graph = builder.compile()

# Example usage
result = graph.invoke({"user_query": "What is the total sales for each product?"})
print(result)

In [ ]:
print(result['sql_query'])

In [ ]:
print(result['sql_explanation'])

In [ ]:
pip install -qU langchain-huggingface

In [ ]:
# @title **Architecture #3: Router**

from typing import Annotated, Literal, TypedDict
from langchain_groq import ChatGroq
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.vectorstores.in_memory import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

# HuggingFace 384차원 임베딩 모델 설정
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# useful to generate SQL query
model_low_temp = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.1)
# useful to generate natural language outputs
model_low_temp = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.1)

class State(TypedDict):
    # to track conversation history
    messages: Annotated[list, add_messages]
    # input
    user_query: str
    # output
    domain: Literal["records", "insurance"]
    documents: list[Document]
    answer: str

class Input(TypedDict):
    user_query: str

class Output(TypedDict):
    documents: list[Document]
    answer: str

# Sample documents for testing
sample_docs = [
    Document(page_content="Patient medical record...", metadata={"domain": "records"}),
    Document(
        page_content="Insurance policy details...", metadata={"domain": "insurance"}
    ),
]

# Initialize vector stores
medical_records_store = InMemoryVectorStore.from_documents(sample_docs, embeddings)
medical_records_retriever = medical_records_store.as_retriever()

insurance_faqs_store = InMemoryVectorStore.from_documents(sample_docs, embeddings)
insurance_faqs_retriever = insurance_faqs_store.as_retriever()

router_prompt = SystemMessage(
    """You need to decide which domain to route the user query to. You have two domains to choose from:
- records: contains medical records of the patient, such as diagnosis, treatment, and prescriptions.
- insurance: contains frequently asked questions about insurance policies, claims, and coverage.

Output only the domain name."""
)

def router_node(state: State) -> State:
    user_message = HumanMessage(state["user_query"])
    messages = [router_prompt, *state["messages"], user_message]
    res = model_low_temp.invoke(messages)
    return {
        "domain": res.content,
        # update conversation history
        "messages": [user_message, res],
    }

def pick_retriever(
    state: State,
) -> Literal["retrieve_medical_records", "retrieve_insurance_faqs"]:
    if state["domain"] == "records":
        return "retrieve_medical_records"
    else:
        return "retrieve_insurance_faqs"

def retrieve_medical_records(state: State) -> State:
    documents = medical_records_retriever.invoke(state["user_query"])
    return {
        "documents": documents,
    }

def retrieve_insurance_faqs(state: State) -> State:
    documents = insurance_faqs_retriever.invoke(state["user_query"])
    return {
        "documents": documents,
    }

medical_records_prompt = SystemMessage(
    "You are a helpful medical chatbot, who answers questions based on the patient's medical records, such as diagnosis, treatment, and prescriptions."
)

insurance_faqs_prompt = SystemMessage(
    "You are a helpful medical insurance chatbot, who answers frequently asked questions about insurance policies, claims, and coverage."
)

def generate_answer(state: State) -> State:
    if state["domain"] == "records":
        prompt = medical_records_prompt
    else:
        prompt = insurance_faqs_prompt
    messages = [
        prompt,
        *state["messages"],
        HumanMessage(f"Documents: {state['documents']}"),
    ]
    res = model_high_temp.invoke(messages)
    return {
        "answer": res.content,
        # update conversation history
        "messages": res,
    }

builder = StateGraph(State, input=Input, output=Output)
builder.add_node("router", router_node)
builder.add_node("retrieve_medical_records", retrieve_medical_records)
builder.add_node("retrieve_insurance_faqs", retrieve_insurance_faqs)
builder.add_node("generate_answer", generate_answer)
builder.add_edge(START, "router")
builder.add_conditional_edges("router", pick_retriever)
builder.add_edge("retrieve_medical_records", "generate_answer")
builder.add_edge("retrieve_insurance_faqs", "generate_answer")
builder.add_edge("generate_answer", END)

graph = builder.compile()

# Example usage
input = {"user_query": "Am I covered for COVID-19 treatment?"}
for chunk in graph.stream(input):
    print(chunk)

In [ ]:
input = {"user_query": "Am I covered for COVID-19 treatment?"}
for chunk in graph.stream(input):
    print(chunk)